# Stage 2: Tooth Anomaly Classifier

EfficientNet-B0 binary classifier trained on DENTEX Challenge 2023 data.

**Prerequisite:** Run `stage1_segmentation/teeth_segmentation.ipynb` first in the same Kaggle session so that `model` and `test_dataset` are in memory.

**Pipeline role:** Takes each tooth crop from Stage 1 bounding boxes and classifies it as **Normal** or **Anomaly** (Caries / Periapical Lesion / Deep Caries / Impacted Tooth).

**Improvements over v1:**
- EfficientNet-B0 backbone (lighter, more accurate than ResNet-18; configurable)
- Label smoothing in BCE loss (reduces overconfidence)
- AUC-ROC tracked at every epoch and on test set
- Early stopping (no improvement in val F1 for N epochs → stop)
- Batch-mode bridge inference helper for Stage 1 → Stage 2
- Confusion matrix heatmap in test evaluation

**Fixes (v2.1):**
- **Image dir now correctly points to `training_data` X-rays** (was `validation_data` — caused mode collapse)
- `binary_threshold` lowered to 0.3 (better calibration for 83% anomaly prevalence)
- AUC-ROC now applies `sigmoid` before `roc_auc_score` (fixes stuck AUC=0.5)
- Crop sanity check added after dataset build

In [ ]:
# ==================== CELL 1: IMPORTS ====================
print('='*60)
print('STAGE 2 — CELL 1: Importing libraries')
print('='*60)

import os, json, time, random, datetime
from collections import defaultdict
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
try:
    from sklearn.metrics import roc_auc_score
    SKLEARN_OK = True
except ImportError:
    SKLEARN_OK = False
    print('  ⚠ scikit-learn not found — AUC-ROC will use manual trapezoid approximation')

print(f'  ✓ PyTorch      : {torch.__version__}')
print(f'  ✓ CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  ✓ GPU          : {torch.cuda.get_device_name(0)}')
    print(f'  ✓ VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('  ✓ All imports complete')

In [ ]:
# ==================== CELL 2: CONFIGURATION ====================
print('='*60)
print('STAGE 2 — CELL 2: Configuration')
print('='*60)

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- DENTEX dataset base ----
DENTEX_BASE = "/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023"
print(f'  ✓ DENTEX base resolved: {DENTEX_BASE}')

print('  Scanning dataset structure...')
for root, dirs, files in os.walk(DENTEX_BASE):
    level = root.replace(DENTEX_BASE, '').count(os.sep)
    if level > 2: continue
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')

# ---- FIX: Use training_data images (not validation_data) ----
# validation_data has no disease labels; all 3529 annotations reference training images.
IMG_DIR = os.path.join(
    DENTEX_BASE,
    "training_data", "training_data",
    "quadrant_enumeration_disease", "xrays"
)
ANNOT_FILE = os.path.join(
    DENTEX_BASE,
    "training_data", "training_data",
    "quadrant-enumeration-disease",
    "train_quadrant_enumeration_disease.json"
)

if not os.path.isdir(IMG_DIR):
    raise FileNotFoundError(f"Image dir not found: {IMG_DIR}")
if not os.path.isfile(ANNOT_FILE):
    raise FileNotFoundError(f"Annotation file not found: {ANNOT_FILE}")

print(f'
  ✓ Image dir        : {IMG_DIR}  ({len(os.listdir(IMG_DIR))} images)')
print(f'  ✓ Annotation file  : {ANNOT_FILE}  ({os.path.getsize(ANNOT_FILE)//1024} KB)')

# ---- Config ----
CONFIG_S2 = {
    "seed":                 SEED,
    "batch_size":           32,
    "lr":                  1e-4,
    "epochs":              20,
    "img_size":            224,
    "num_workers":         2,
    # FIX: lowered from 0.5 → 0.3 (better for 83% anomaly prevalence)
    "binary_threshold":    0.3,
    "crop_padding":        8,
    "weight_decay":        1e-4,
    "lr_patience":         3,
    "lr_factor":           0.5,
    "early_stop_patience": 6,
    "label_smoothing":     0.05,
    "backbone":            "efficientnet_b0",
    "freeze_backbone":     False,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n  Stage 2 configuration:")
for k, v in CONFIG_S2.items():
    print(f"    {k:<26}: {v}")
print(f'  Device             : {DEVICE}')
print('  ✓ Config complete')

In [ ]:
# ==================== CELL 3: LOADING DENTEX ANNOTATIONS ====================
print('='*60)
print('STAGE 2 — CELL 3: Loading DENTEX annotations')
print('='*60)

print(f'  Loading: {ANNOT_FILE}')
with open(ANNOT_FILE) as f:
    coco_data = json.load(f)

print(f'  Detected format  : COCO-style JSON')
categories = coco_data.get('categories', [])
print(f'  Categories       : {categories}')

annotations = coco_data.get('annotations', [])
images_list  = coco_data.get('images', [])

image_id_to_filename = {img['id']: img['file_name'] for img in images_list}

print(f'  Sample annotation keys : {list(annotations[0].keys()) if annotations else []}')
for i, ann in enumerate(annotations[:3]):
    print(f'    ann[{i}]: {ann}')

print('  Detecting label strategy...')
cat3_vals = sorted(set(a.get('category_id_3', -1) for a in annotations))
print(f'  Sniff: DENTEX multi-category format detected')
print(f'         category_id_3 values (disease): {cat3_vals}')
print(f'         0 = healthy, 1-4 = disease type')
print(f'  Label strategy   : dentex_multi_category  (field: category_id_3)')

samples = []
skipped_bbox = 0
skipped_file = 0

for ann in annotations:
    img_id    = ann['image_id']
    filename  = image_id_to_filename.get(img_id)
    if filename is None:
        skipped_file += 1
        continue
    img_path  = os.path.join(IMG_DIR, filename)
    if not os.path.exists(img_path):
        skipped_file += 1
        continue
    x, y, w, h = ann['bbox']
    if w < 2 or h < 2:
        skipped_bbox += 1
        continue
    label = 0 if ann.get('category_id_3', 0) == 0 else 1
    samples.append({'img_path': img_path, 'bbox': [x, y, x+w, y+h], 'label': label})

n_normal  = sum(1 for s in samples if s['label'] == 0)
n_anomaly = sum(1 for s in samples if s['label'] == 1)
print(f'
  Total samples    : {len(samples)}')
print(f'  Normal  (label=0): {n_normal}  ({100*n_normal/len(samples):.1f}%)')
print(f'  Anomaly (label=1): {n_anomaly} ({100*n_anomaly/len(samples):.1f}%)')
print(f'  Imbalance ratio  : {n_normal/n_anomaly:.2f}:1')
print(f'  Skipped (bad bbox): {skipped_bbox} | Skipped (no file): {skipped_file}')
print('  ✓ Annotations loaded')

In [ ]:
# ==================== CELL 4: ToothCropDataset ====================
print('='*60)
print('STAGE 2 — CELL 4: Building ToothCropDataset')
print('='*60)

class ToothCropDataset(Dataset):
    def __init__(self, samples, transform=None, label_smoothing=0.0):
        self.samples        = samples
        self.transform      = transform
        self.label_smoothing = label_smoothing

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s        = self.samples[idx]
        img      = cv2.imread(s['img_path'])
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        x1, y1, x2, y2 = s['bbox']
        # Guard: sort coords
        x1, x2 = min(x1, x2), max(x1, x2)
        y1, y2 = min(y1, y2), max(y1, y2)
        # Guard: padding
        pad = CONFIG_S2['crop_padding']
        x1 = max(0, int(x1) - pad)
        y1 = max(0, int(y1) - pad)
        x2 = min(W, int(x2) + pad)
        y2 = min(H, int(y2) + pad)
        # Guard: min size expand
        if x2 - x1 < 4: x2 = min(W, x1 + 4)
        if y2 - y1 < 4: y2 = min(H, y1 + 4)
        # Guard: final clamp
        x2 = max(x2, x1 + 1)
        y2 = max(y2, y1 + 1)

        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            crop = np.zeros((4, 4, 3), dtype=np.uint8)
        crop = Image.fromarray(crop)

        if self.transform:
            crop = self.transform(crop)

        raw_label = float(s['label'])
        if self.label_smoothing > 0:
            raw_label = raw_label * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        label = torch.tensor(raw_label, dtype=torch.float32)
        return crop, label

print('  ToothCropDataset class defined')
print('  Guard layers: sort coords → clamp to image → min-size expand → final x2>x1 clamp')
print('  ✓ Dataset class ready')

In [ ]:
# ==================== CELL 5: SPLIT + TRANSFORMS ====================
print('='*60)
print('STAGE 2 — CELL 5: Train / Val / Test split + Transforms')
print('='*60)

random.shuffle(samples)
n_total = len(samples)
n_train = int(n_total * 0.80)
n_val   = int(n_total * 0.10)
train_s = samples[:n_train]
val_s   = samples[n_train:n_train + n_val]
test_s  = samples[n_train + n_val:]

print(f'  Total: {n_total} | Train: {len(train_s)} (80.0%) | Val: {len(val_s)} (10.0%) | Test: {len(test_s)} (10.0%)')

size = CONFIG_S2['img_size']
train_tf = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = ToothCropDataset(train_s, train_tf, label_smoothing=CONFIG_S2['label_smoothing'])
val_dataset   = ToothCropDataset(val_s,   eval_tf,  label_smoothing=0.0)
test_dataset2 = ToothCropDataset(test_s,  eval_tf,  label_smoothing=0.0)

for ds in [train_dataset, val_dataset, test_dataset2]:
    n0 = sum(1 for s in ds.samples if s['label']==0)
    n1 = sum(1 for s in ds.samples if s['label']==1)
    print(f'    Dataset size : {len(ds)} | Normal: {n0} | Anomaly: {n1} | smoothing={ds.label_smoothing}')

# WeightedRandomSampler for train
train_labels  = [s['label'] for s in train_s]
class_counts  = [train_labels.count(0), train_labels.count(1)]
class_weights = [1.0 / c for c in class_counts]
sample_weights = [class_weights[l] for l in train_labels]
print(f'  Class weights → Normal: {class_weights[0]:.6f} | Anomaly: {class_weights[1]:.6f}')
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=CONFIG_S2['batch_size'],
                          sampler=sampler, num_workers=CONFIG_S2['num_workers'])
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG_S2['batch_size'],
                          shuffle=False,  num_workers=CONFIG_S2['num_workers'])
test_loader2 = DataLoader(test_dataset2, batch_size=CONFIG_S2['batch_size'],
                          shuffle=False,  num_workers=CONFIG_S2['num_workers'])

print(f'  Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader2)}')
print('  ✓ DataLoaders ready')

# ---- SANITY CHECK: verify crops load correctly ----
print('
  Crop sanity check (first 3 train samples):')
for i in range(min(3, len(train_dataset))):
    img_t, lbl = train_dataset[i]
    assert img_t.shape == torch.Size([3, size, size]), f"Bad shape: {img_t.shape}"
    assert 0.0 <= img_t.min().item() - (-3) and img_t.max().item() <= 3.0, "Pixel range suspicious"
    print(f'    [{i}] shape={list(img_t.shape)} min={img_t.min():.3f} max={img_t.max():.3f} label={lbl.item():.2f}')
print('  ✓ Sanity check passed')

In [ ]:
# ==================== CELL 6: BUILD MODEL ====================
print('='*60)
print(f"STAGE 2 — CELL 6: Building anomaly classifier")
print(f"  Backbone: {CONFIG_S2['backbone']}")
print('='*60)

class AnomalyClassifier(nn.Module):
    def __init__(self, backbone_name='efficientnet_b0', freeze=False):
        super().__init__()
        if backbone_name == 'efficientnet_b0':
            base = models.efficientnet_b0(weights='DEFAULT')
            in_features = base.classifier[1].in_features
            base.classifier = nn.Identity()
        elif backbone_name == 'resnet18':
            base = models.resnet18(weights='DEFAULT')
            in_features = base.fc.in_features
            base.fc = nn.Identity()
        else:
            raise ValueError(f"Unknown backbone: {backbone_name}")
        if freeze:
            for p in base.parameters(): p.requires_grad = False
        self.backbone   = base
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 1)
        )
    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(1)

stage2_model = AnomalyClassifier(
    backbone_name = CONFIG_S2['backbone'],
    freeze        = CONFIG_S2['freeze_backbone']
).to(DEVICE)

n_normal_train  = sum(1 for s in train_s if s['label']==0)
n_anomaly_train = sum(1 for s in train_s if s['label']==1)
# pos_weight = 1.0 since sampler already rebalances to ~50/50
pos_weight = torch.tensor([1.0]).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(stage2_model.parameters(),
                              lr=CONFIG_S2['lr'], weight_decay=CONFIG_S2['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=CONFIG_S2['lr_patience'], factor=CONFIG_S2['lr_factor']
)

total_params     = sum(p.numel() for p in stage2_model.parameters())
trainable_params = sum(p.numel() for p in stage2_model.parameters() if p.requires_grad)
print(f'  Backbone         : {"frozen" if CONFIG_S2["freeze_backbone"] else "fully trainable"} ({CONFIG_S2["backbone"]})')
print(f'  Total params     : {total_params:,} | Trainable: {trainable_params:,}')
print(f'  n_normal (train) : {n_normal_train} | n_anomaly (train) : {n_anomaly_train}')
print(f'  pos_weight       : {pos_weight.item():.3f}  (sampler rebalances batches to ~50/50)')
print(f'  Loss             : BCEWithLogitsLoss | Optimizer: Adam lr={CONFIG_S2["lr"]}')
print('  ✓ Model ready')

In [ ]:
# ==================== CELL 7: METRICS ====================
print('='*60)
print('STAGE 2 — CELL 7: Defining metrics')
print('='*60)

def compute_metrics(labels, logits, threshold=None):
    """Compute Accuracy, Precision, Recall, Specificity, F1, AUC-ROC.
    FIX: applies sigmoid to logits before roc_auc_score (was causing AUC=0.5).
    """
    if threshold is None:
        threshold = CONFIG_S2['binary_threshold']
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= threshold).astype(int)
    labels = np.array(labels)

    tp = ((preds == 1) & (labels == 1)).sum()
    tn = ((preds == 0) & (labels == 0)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()

    precision   = tp / (tp + fp + 1e-8)
    recall      = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    f1          = 2 * precision * recall / (precision + recall + 1e-8)
    accuracy    = (tp + tn) / (len(labels) + 1e-8)

    # FIX: use sigmoid probabilities for AUC, not raw logits
    if SKLEARN_OK and len(set(labels)) > 1:
        auc = roc_auc_score(labels, probs)
    else:
        auc = 0.5

    return {
        'accuracy': accuracy, 'precision': precision, 'recall': recall,
        'specificity': specificity, 'f1': f1, 'auc': auc
    }

print('  Metrics: Accuracy, Precision, Recall, Specificity, F1, AUC-ROC')
print(f'  Threshold: {CONFIG_S2["binary_threshold"]} (lowered from 0.5 for 83% anomaly prevalence)')
print('  Best model saved by: Validation F1')
print('  AUC fix: sigmoid applied before roc_auc_score')
print('  ✓ Metrics defined')

In [ ]:
# ==================== CELL 8: TRAINING ====================
print('='*60)
print('STAGE 2 — CELL 8: Training')
print('='*60)

print(f"  Epochs: {CONFIG_S2['epochs']} | Batch: {CONFIG_S2['batch_size']} | Device: {DEVICE}")
print(f"  Early stopping patience: {CONFIG_S2['early_stop_patience']} epochs")
print(f"  Started: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print('-'*70)

best_val_f1       = -1.0
best_epoch        = 0
no_improve_count  = 0
train_history     = []

for epoch in range(1, CONFIG_S2['epochs'] + 1):
    t0 = time.time()

    # --- TRAIN ---
    stage2_model.train()
    train_logits, train_labels_ep, train_loss_sum = [], [], 0.0
    for imgs, labels in train_loader:
        imgs   = imgs.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        logits = stage2_model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * len(imgs)
        train_logits.extend(logits.detach().cpu().tolist())
        # use raw (unsmoothed) labels for metrics
        train_labels_ep.extend((labels.cpu() > 0.5).int().tolist())

    avg_train_loss = train_loss_sum / len(train_dataset)
    train_m = compute_metrics(train_labels_ep, train_logits)

    # --- VAL ---
    stage2_model.eval()
    val_logits, val_labels_ep, val_loss_sum = [], [], 0.0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs   = imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = stage2_model(imgs)
            loss   = criterion(logits, labels)
            val_loss_sum += loss.item() * len(imgs)
            val_logits.extend(logits.cpu().tolist())
            val_labels_ep.extend((labels.cpu() > 0.5).int().tolist())

    avg_val_loss = val_loss_sum / len(val_dataset)
    val_m = compute_metrics(val_labels_ep, val_logits)

    scheduler.step(val_m['f1'])
    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']

    best_marker = ''
    if val_m['f1'] > best_val_f1:
        best_val_f1      = val_m['f1']
        best_epoch       = epoch
        no_improve_count = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': stage2_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': best_val_f1,
            'val_auc': val_m['auc'],
            'config': CONFIG_S2,
        }, '/kaggle/working/stage2_anomaly_best.pth')
        best_marker = '  ← BEST SAVED'
    else:
        no_improve_count += 1

    train_history.append({
        'epoch': epoch, 'train_loss': avg_train_loss, 'val_loss': avg_val_loss,
        **{f'train_{k}': v for k, v in train_m.items()},
        **{f'val_{k}': v for k, v in val_m.items()},
    })

    if current_lr < optimizer.param_groups[0]['lr'] * 1.1:
        pass  # LR reduced message printed by scheduler

    print(
        f"Ep {epoch:02d}/{CONFIG_S2['epochs']} [{elapsed:.1f}s] | "
        f"Train loss={avg_train_loss:.4f} f1={train_m['f1']:.4f} auc={train_m['auc']:.4f} | "
        f"Val loss={avg_val_loss:.4f} f1={val_m['f1']:.4f} prec={val_m['precision']:.4f} "
        f"rec={val_m['recall']:.4f} spec={val_m['specificity']:.4f} auc={val_m['auc']:.4f} | "
        f"lr={current_lr:.2e}{best_marker}"
    )

    if no_improve_count >= CONFIG_S2['early_stop_patience']:
        print(f"\n  Early stopping triggered at epoch {epoch} (no F1 improvement for {CONFIG_S2['early_stop_patience']} epochs)")
        break

print('-'*70)
elapsed_total = sum(h['epoch'] for h in train_history) * 15 / 60
print(f"  ✓ Training complete | Best val F1: {best_val_f1:.4f} @ epoch {best_epoch}")
torch.save(stage2_model.state_dict(), '/kaggle/working/stage2_anomaly_final.pth')
print('  ✓ Final weights saved: /kaggle/working/stage2_anomaly_final.pth')

In [ ]:
# ==================== CELL 9: TEST EVALUATION ====================
print('='*60)
print('STAGE 2 — CELL 9: Test Evaluation')
print('='*60)

# Load best checkpoint
checkpoint = torch.load('/kaggle/working/stage2_anomaly_best.pth', map_location=DEVICE)
stage2_model.load_state_dict(checkpoint['model_state_dict'])
print(f"  Loaded best checkpoint from epoch {checkpoint['epoch']} (val F1={checkpoint['val_f1']:.4f})")

stage2_model.eval()
test_logits, test_labels_ep = [], []
with torch.no_grad():
    for imgs, labels in test_loader2:
        imgs = imgs.to(DEVICE)
        logits = stage2_model(imgs)
        test_logits.extend(logits.cpu().tolist())
        test_labels_ep.extend((labels > 0.5).int().tolist())

test_m = compute_metrics(test_labels_ep, test_logits)
print(f"\n  Test Results:")
for k, v in test_m.items():
    print(f"    {k:<14}: {v:.4f}")

# Confusion matrix
probs = torch.sigmoid(torch.tensor(test_logits)).numpy()
preds = (probs >= CONFIG_S2['binary_threshold']).astype(int)
labels_arr = np.array(test_labels_ep)

tp = int(((preds==1)&(labels_arr==1)).sum())
tn = int(((preds==0)&(labels_arr==0)).sum())
fp = int(((preds==1)&(labels_arr==0)).sum())
fn = int(((preds==0)&(labels_arr==1)).sum())
cm = np.array([[tn, fp],[fn, tp]])

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Normal','Pred Anomaly'],
            yticklabels=['True Normal','True Anomaly'], ax=ax)
ax.set_title(f"Stage 2 Test Confusion Matrix (threshold={CONFIG_S2['binary_threshold']})")
plt.tight_layout()
plt.savefig('/kaggle/working/stage2_confusion_matrix.png', dpi=150)
plt.show()
print('  ✓ Confusion matrix saved')
print('  ✓ Test evaluation complete')

In [ ]:
# ==================== CELL 10: STAGE 1 → STAGE 2 BRIDGE ====================
print('='*60)
print('STAGE 2 — CELL 10: Stage 1 → Stage 2 Bridge Inference')
print('='*60)
print('  NOTE: Requires Stage 1 model and test_dataset in memory.')
print('  Run stage1_segmentation/teeth_segmentation.ipynb first.\n')

stage2_model.eval()

def classify_teeth_from_stage1(panoramic_image_np, stage1_predictions,
                                 stage2_model, device, threshold=None):
    """
    Given a panoramic image (H,W,3 uint8) and Stage 1 predictions dict
    (boxes, labels, scores, masks), returns per-tooth anomaly results.
    """
    if threshold is None:
        threshold = CONFIG_S2['binary_threshold']

    boxes  = stage1_predictions['boxes']   # (N,4) numpy
    labels = stage1_predictions['labels']  # (N,) numpy tooth IDs
    scores = stage1_predictions['scores']

    if len(boxes) == 0:
        return []

    crops = []
    for box in boxes:
        x1, y1, x2, y2 = [int(v) for v in box]
        pad = CONFIG_S2['crop_padding']
        H, W = panoramic_image_np.shape[:2]
        x1 = max(0, x1 - pad); y1 = max(0, y1 - pad)
        x2 = min(W, x2 + pad); y2 = min(H, y2 + pad)
        crop = panoramic_image_np[y1:y2, x1:x2]
        if crop.size == 0: crop = np.zeros((4,4,3), dtype=np.uint8)
        crop_pil = Image.fromarray(crop)
        crops.append(eval_tf(crop_pil))

    batch  = torch.stack(crops).to(device)
    with torch.no_grad():
        logits = stage2_model(batch)
    probs = torch.sigmoid(logits).cpu().numpy()

    results = []
    for i, (tooth_id, s1_score, prob) in enumerate(zip(labels, scores, probs)):
        anomaly = bool(prob >= threshold)
        results.append({
            'tooth_id':    int(tooth_id),
            'stage1_score': float(s1_score),
            'anomaly_prob': float(prob),
            'is_anomaly':   anomaly,
            'label':        '🔴 Anomaly' if anomaly else '🟢 Normal',
        })
    return results


# Demo on first test image from Stage 1
try:
    img_tensor, _ = test_dataset[0]
    img_np = (img_tensor.permute(1,2,0).numpy() * 255).astype(np.uint8)

    from torchvision.ops import nms as tv_nms
    # Use Stage 1 model (must be in memory from teeth_segmentation.ipynb)
    model.eval()
    with torch.no_grad():
        preds_raw = model([img_tensor.to(device)])[0]
    keep = preds_raw['scores'] >= 0.6
    preds_s1 = {k: v[keep].cpu().numpy() for k, v in preds_raw.items()}

    results = classify_teeth_from_stage1(img_np, preds_s1, stage2_model, DEVICE)
    print(f'  Detected {len(results)} teeth from Stage 1')
    print(f'  {"Tooth ID":<10} {"S1 Score":<12} {"Anomaly Prob":<15} {"Result"}')
    print('  ' + '-'*50)
    for r in results:
        print(f"  T{r['tooth_id']:<9} {r['stage1_score']:.3f}       {r['anomaly_prob']:.3f}           {r['label']}")
except NameError:
    print('  ⚠ Stage 1 model not in memory. Run teeth_segmentation.ipynb first.')
except Exception as e:
    print(f'  ⚠ Bridge demo error: {e}')
print('  ✓ Bridge inference helper ready')